In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
(train_images, train_labels), (_, _) = tf.keras.datasets.cifar10.load_data()

filter = np.where((train_labels ==0) | (train_labels == 1) | (train_labels == 2))
train_images = train_images[filter[0]].astype('float32')
train_images = (train_images - 127.5) / 127.5


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
print(train_images.shape)
print(train_images[0].shape)

plt.imshow(train_images[10110])
plt.show()


In [ ]:
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (3,3), padding='same', input_shape=(32,32,3)),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(128, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(128, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(256, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(256, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(256, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2DTranspose(512, (4,4), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),
    ])
    return model


In [ ]:
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(256*4*4, input_dim=100),
        layers.LeakyReLU(alpha=0.2),
        layers.Reshape((4,4,256)),

        layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2D(256, (3,3), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'),
        layers.LeakyReLU(alpha=0.2),

        layers.Dense(3,activation='tanh')
    ])
    return model


In [ ]:
generator = build_generator()
discriminator = build_discriminator()

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        disc_loss = cross_entropy(tf.ones_like(real_output), real_output) + \
                    cross_entropy(tf.zeros_like(fake_output), fake_output)

    gradients_of_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_gen, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_disc, discriminator.trainable_variables))

    return gen_loss, disc_loss

def generate_images(epoch):
    predictions = generator(tf.random.normal([16, 100]), training=False)
    fig = plt.figure(figsize=(4,4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow((predictions[i] * 127.5 + 127.5).numpy().astype('uint8'))
        plt.axis('off')
    plt.show()
    plt.close()


In [ ]:
EPOCHS = 1000
BATCH_SIZE = 256

train_dataset = tf.data.Dataset.from_tensor_slices(train_images) \
    .shuffle(6000).batch(BATCH_SIZE)

for epoch in range(EPOCHS):
    for image_batch in train_dataset:
        gen_loss, disc_loss = train_step(image_batch)
    if(epoch%10==0):
      generate_images(epoch)
    if epoch%100==0:
      generator.save(f'generator_cifar10_{epoch}epochs.keras')
      discriminator.save(f'discriminator_cifar10_{epoch}epochs.keras')
    print(f'Epoch {epoch+1} | Gen loss: {gen_loss:.4f} | Disc loss: {disc_loss:.4f}')


In [ ]:
generator.save('generator_cifar10_1000epochs.keras')
discriminator.save('discriminator_cifar10_1000epochs.keras')